# 03 - Bronce: ingesta en streaming (Auto Loader)

**Tarea `bronce_streaming`.** Structured Streaming con Auto Loader (`cloudFiles`, CSV con esquema explícito de texto, `_rescued_data`) hacia `bronze.ais_raw` (Delta append-only, `delta.appendOnly=true`).

- **Checkpoint:** `/Volumes/<catálogo>/ops/checkpoints/bronze_ais_raw` registra archivos procesados y offsets.
- **Disparo:** `availableNow`: procesa todo lo pendiente y termina (apto para job programado).
- **Tolerancia a fallos:** el sumidero Delta es idempotente por `batchId`; ver `docs/evidencias/06 evidencia_tolerancia_fallos.md`.

In [0]:
import os
import sys

# El código del pipeline vive en ../src (misma librería probada en local; ver README).
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from oceanwatch.lakehouse import get_lakehouse
from oceanwatch.notebook_params import fechas, widget

CATALOG = widget(dbutils, "catalogo", "oceanwatch_g06")
lh = get_lakehouse(spark, catalog=CATALOG)

In [0]:
import json

from oceanwatch.streaming import run_bronze

resumen = run_bronze(lh)
print(json.dumps(resumen, indent=2))
display(spark.sql(f"DESCRIBE HISTORY {lh.name('bronze', 'ais_raw')}").limit(5))

{
  "query": "bronze_ais_raw",
  "lotes_con_datos": 0,
  "filas_entrada": 0,
  "ids_lote": []
}


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-19T17:35:58.912Z,75682274564512,arangurenmarita@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4a459739-b615-4621-9094-259bc6f0b837, epochId -> 2, statsOnLoad -> true)",null,List(1301092897847358),84626500-1501-47e8-98eb-8d483485fd8b,1119-164729-fys2bnqo-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 7794117, numOutputBytes -> 171813111, numAddedFiles -> 3)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
2,2026-11-19T17:21:03.757Z,75682274564512,arangurenmarita@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4a459739-b615-4621-9094-259bc6f0b837, epochId -> 1, statsOnLoad -> true)",null,List(1301092897847358),b3746e0b-6346-4e0d-84e5-380856cbca81,1119-164729-fys2bnqo-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 8791124, numOutputBytes -> 195981148, numAddedFiles -> 3)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
1,2026-11-19T16:02:58.572Z,75682274564512,arangurenmarita@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4a459739-b615-4621-9094-259bc6f0b837, epochId -> 0, statsOnLoad -> true)",null,List(956002230196489),d8cc607d-24b3-44a9-b486-811a6e575cd4,1119-152712-1459rda7-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 8920444, numOutputBytes -> 196925854, numAddedFiles -> 3)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
0,2026-11-19T15:35:38.770Z,75682274564512,arangurenmarita@gmail.com,CREATE TABLE,"Map(partitionBy -> [""ingestion_date""], clusterBy -> [], description -> Bronce: AIS crudo e inmutable tal como llegó del feed diario NOAA (todas las columnas como texto). Append-only., isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.appendOnly"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""869d8c03-63fd-47a2-9f0d-dd7c43421f66"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a28ea482-9f82-4ded-b286-e49734d07bea"",""delta.enableInCommitTimestamps"":""true"",""oceanwatch.capa"":""bronce"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-bec76be8-75a4-4dca-97fb-558c3b616e6a""}, statsOnLoad -> false)",null,List(956002230196481),4bbb67c2-e629-4060-934f-3b47df884978,1119-152712-1459rda7-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
